# G2 segment inference

Run from the `prebi_v1` project root with the project `.venv` kernel. This notebook uses the four-model direct baseline: one scope classifier and three skill classifiers. It predicts one selected document from text-only test-split columns; it never loads human annotation targets or sends text to an external provider. Set the document selector to the same value in `G2_Demo.ipynb`. Use the test split only for an approved pilot or final inference, not model or prompt tuning.

## Local environment configuration

Copy `.env.example` to `.env` and adjust `PREBI_DATA_ROOT` to an approved absolute directory outside the project. `.env` is local-only and ignored by Git. The notebook loads it from the project root with `load_dotenv(..., override=False)`, so an existing shell environment value takes precedence.

```dotenv
PREBI_DATA_ROOT="${HOME}/prebi-protected-data"
```

## Imports and runtime configuration

In [2]:
import gc
import json
import os
import tempfile
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

from reflection_assessment_feedback import (
    EXPERT_RUBRIC,
    IntermediateAnalysis,
    ReflectionDocument,
    ReflectionSegment,
    SegmentAnalysis,
    make_generation_request,
)

PROJECT_ROOT = Path.cwd().resolve()
assert PROJECT_ROOT.name == "prebi_v1", "Run from the prebi_v1 project root."
load_dotenv(PROJECT_ROOT / ".env", override=False)
SPLIT_PATH = PROJECT_ROOT / "data" / "provisional_segment_splits" / "provisional_test_modeling_wide.csv"
CHECKPOINT_DIR = PROJECT_ROOT / "artifacts" / "segment-classification"
MODEL_NAME = "deepset/gbert-base"
MODEL_REVISION = None
MAX_LENGTH = 256
BATCH_SIZE = 8
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
SKILL_DIMENSIONS = {"situationserfassung": "SW", "analyse": "UA", "konsequenzen": "HA"}
MODEL_LABELS = {"scope": 2, **{f"direct-{skill}": 4 for skill in SKILL_DIMENSIONS}}
print(f"Inference device: {DEVICE}")

Inference device: mps


## Resolve protected paths and target document

Use the same `DOCUMENT_INDEX` as the G2 notebook (or set `DOCUMENT_ID` explicitly in both). `PREBI_DATA_ROOT` must be an approved absolute directory outside this source tree. The fixed output filename serves the selected document only; the writer will not replace an artifact for a different document.

In [3]:
DOCUMENT_INDEX = 0
DOCUMENT_ID = None
protected_root_value = os.getenv("PREBI_DATA_ROOT")
OUTPUT_PATH = None
if protected_root_value:
    protected_root = Path(protected_root_value).expanduser()
    if not protected_root.is_absolute():
        raise ValueError("PREBI_DATA_ROOT must be an absolute path outside the project.")
    protected_root = protected_root.resolve()
    if protected_root == PROJECT_ROOT or PROJECT_ROOT in protected_root.parents:
        raise ValueError("PREBI_DATA_ROOT must be outside the project source tree.")
    output_dir = protected_root / "g2"
    output_dir.mkdir(parents=True, exist_ok=True, mode=0o700)
    OUTPUT_PATH = output_dir / "predicted-analysis.json"
else:
    print("Set PREBI_DATA_ROOT to an approved protected directory before writing.")

document_ids = sorted(pd.read_csv(SPLIT_PATH, usecols=["document_id"])["document_id"].dropna().astype(str).unique())
if DOCUMENT_ID is None:
    if not 0 <= DOCUMENT_INDEX < len(document_ids):
        raise IndexError("DOCUMENT_INDEX is outside the available document list.")
    selected_document_id = document_ids[DOCUMENT_INDEX]
else:
    selected_document_id = str(DOCUMENT_ID)
    if selected_document_id not in document_ids:
        raise ValueError("DOCUMENT_ID is not in the selected split.")
print("Target document selected.")

Target document selected.


## Load document segments for inference

Read only IDs, text, and source order. Candidate annotation rows may duplicate a segment; require their text and order to agree before selecting one copy. The model sees one row per actual segment, without any gold targets.

In [4]:
reflection_rows = pd.read_csv(
    SPLIT_PATH, usecols=["document_id", "segment_id", "text", "segment_order"]
)
reflection_rows = reflection_rows.loc[reflection_rows["document_id"].astype(str) == selected_document_id].copy()
if reflection_rows[["segment_id", "text", "segment_order"]].isna().any().any():
    raise ValueError("Selected document contains missing segment IDs, text, or order.")
reflection_rows["segment_id"] = reflection_rows["segment_id"].astype(str)
reflection_rows["text"] = reflection_rows["text"].astype(str)
reflection_rows["segment_order"] = pd.to_numeric(reflection_rows["segment_order"], errors="raise").astype(int)
consistency = reflection_rows.groupby("segment_id").agg(
    text_variants=("text", "nunique"), order_variants=("segment_order", "nunique")
)
if (consistency > 1).any().any():
    raise ValueError("Candidate copies disagree on segment text or source order.")
document_rows = reflection_rows.drop_duplicates("segment_id").sort_values("segment_order").reset_index(drop=True)
document = ReflectionDocument(
    document_id=selected_document_id,
    segments=[
        ReflectionSegment(segment_id=str(row.segment_id), text=str(row.text), order=int(str(row.segment_order)))
        for row in document_rows.itertuples(index=False)
    ],
)
print(f"Loaded {len(document.segments)} ordered segments (text withheld).")

Loaded 47 ordered segments (text withheld).


## Restore checkpointed model artifacts

The four completed direct-baseline runs must exist in `artifacts/segment-classification/`: `scope` and one `direct-<skill>` run for each skill. A bare `.pt` file is not a completed run. The tokenizer and base configuration match the training notebook (`deepset/gbert-base`, revision `None`, length 256); strict weight loading checks the architecture and label head. Change these constants only together with the training configuration.

In [5]:
RUN_INFERENCE = True


def completed_run(run_name, expected_labels):
    metadata_path = CHECKPOINT_DIR / f"{run_name}.json"
    checkpoint_path = CHECKPOINT_DIR / f"{run_name}.pt"
    if not metadata_path.is_file() or not checkpoint_path.is_file():
        raise FileNotFoundError(f"Complete training first: {run_name} requires both .json and .pt artifacts.")
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if (
        metadata.get("num_labels") != expected_labels
        or Path(metadata.get("checkpoint", "")).name != checkpoint_path.name
        or not metadata.get("history")
    ):
        raise ValueError(f"Incomplete or incompatible checkpoint metadata for {run_name}.")
    return checkpoint_path


def restore_model(run_name, expected_labels):
    config = AutoConfig.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, num_labels=expected_labels)
    if config.model_type != "bert" or config.num_labels != expected_labels:
        raise ValueError(f"Unexpected model configuration for {run_name}.")
    model = AutoModelForSequenceClassification.from_config(config)
    state = torch.load(checkpoints[run_name], map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    return model.to(DEVICE).eval()


checkpoints = {}
tokenizer = None
if RUN_INFERENCE:
    checkpoints = {
        run_name: completed_run(run_name, count)
        for run_name, count in MODEL_LABELS.items()
    }
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    print("Four completed checkpoints and tokenizer are ready.")
else:
    print("Inference disabled; set RUN_INFERENCE = True after training all four direct-baseline models.")

Four completed checkpoints and tokenizer are ready.


## Run batched segment-level prediction

Predict scope for every unique segment, then run the direct skill models only for predicted in-scope segments. Keep logits/probabilities in memory only; the G2 contract stores categorical labels, not raw text or confidence scores.

In [6]:
def predict_logits(run_name, texts):
    if tokenizer is None:
        raise RuntimeError("Enable RUN_INFERENCE and load the tokenizer first.")
    model = restore_model(run_name, MODEL_LABELS[run_name])
    batches = []
    with torch.inference_mode():
        for start in range(0, len(texts), BATCH_SIZE):
            encoded = tokenizer(
                texts[start:start + BATCH_SIZE],
                truncation=True,
                max_length=MAX_LENGTH,
                padding=True,
                return_tensors="pt",
            )
            logits = model(**{key: value.to(DEVICE) for key, value in encoded.items()}).logits
            batches.append(logits.cpu())
    del model
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    return torch.cat(batches)


scope_predictions = []
skill_predictions = {}
if RUN_INFERENCE:
    segment_texts = [segment.text for segment in document.segments]
    scope_logits = predict_logits("scope", segment_texts)
    scope_probabilities = torch.softmax(scope_logits, dim=-1)
    scope_predictions = scope_logits.argmax(dim=-1).tolist()
    in_scope_indices = [index for index, label in enumerate(scope_predictions) if label == 1]
    for skill in SKILL_DIMENSIONS:
        if in_scope_indices:
            skill_logits = predict_logits(f"direct-{skill}", [segment_texts[index] for index in in_scope_indices])
            skill_probabilities = torch.softmax(skill_logits, dim=-1)
            skill_predictions[skill] = dict(zip(in_scope_indices, skill_logits.argmax(dim=-1).tolist(), strict=True))
        else:
            skill_predictions[skill] = {}
    print(f"Predicted scope for {len(scope_predictions)} segments and bands for {len(in_scope_indices)} in-scope segments.")
else:
    print("Prediction skipped.")

Predicted scope for 47 segments and bands for 46 in-scope segments.


## Build `IntermediateAnalysis` from predictions

Scope `0` maps to `out_of_scope` with `N` on all three dimensions. Scope `1` maps to `in_scope` and direct classes `0` through `3` become the corresponding string band on `SW`, `UA`, and `HA`. Validate full ordered coverage against the same G2 request contract used by the generation notebook.

In [7]:
def build_predicted_analysis(document, scope_labels, bands_by_skill):
    if len(scope_labels) != len(document.segments):
        raise ValueError("Scope predictions must cover every segment exactly once.")
    segments = []
    for index, (segment, scope_label) in enumerate(zip(document.segments, scope_labels, strict=True)):
        if scope_label == 0:
            scope = "out_of_scope"
            component_bands = {dimension: "N" for dimension in SKILL_DIMENSIONS.values()}
        elif scope_label == 1:
            scope = "in_scope"
            component_bands = {
                dimension: str(bands_by_skill[skill][index])
                for skill, dimension in SKILL_DIMENSIONS.items()
            }
        else:
            raise ValueError("Scope predictions must be class 0 or 1.")
        segments.append(SegmentAnalysis.model_validate({
            "segment_id": segment.segment_id,
            "scope": scope,
            "component_bands": component_bands,
        }))
    analysis = IntermediateAnalysis.model_validate({"source": "predicted", "segments": segments})
    make_generation_request(condition="G2", document=document, rubric=EXPERT_RUBRIC, analysis=analysis)
    return analysis


predicted_analysis = None
if RUN_INFERENCE:
    predicted_analysis = build_predicted_analysis(document, scope_predictions, skill_predictions)
    print("Predicted analysis passed the G2 document-coverage and label checks.")
else:
    print("Analysis assembly skipped.")

Predicted analysis passed the G2 document-coverage and label checks.


## Write document-scoped JSON to `PREBI_DATA_ROOT/g2/`

Set `WRITE_PREDICTIONS = True` only after reviewing the selected document and approved protected storage. The writer uses a private temporary file and atomic replacement. It refuses to overwrite another document's artifact or substitute human annotations. G2 reads `predicted-analysis.json` from the same protected directory.

In [9]:
def write_predicted_package(output_path, package):
    if output_path.is_file():
        existing = json.loads(output_path.read_text(encoding="utf-8"))
        if existing.get("document_id") != package["document_id"] or existing.get("analysis", {}).get("source") != "predicted":
            raise ValueError("Refusing to replace another document's or a non-predicted artifact.")
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=output_path.parent,
            prefix=".predicted-analysis-", suffix=".tmp", delete=False,
        ) as temporary_file:
            temporary_path = Path(temporary_file.name)
            json.dump(package, temporary_file, ensure_ascii=False, indent=2)
            temporary_file.write("\n")
        os.replace(temporary_path, output_path)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


WRITE_PREDICTIONS = True
prediction_package = None
if WRITE_PREDICTIONS:
    if predicted_analysis is None or OUTPUT_PATH is None:
        raise RuntimeError("Run inference and configure PREBI_DATA_ROOT before writing predictions.")
    prediction_package = {
        "document_id": document.document_id,
        "analysis": predicted_analysis.model_dump(mode="json"),
    }
    if prediction_package["document_id"] != selected_document_id:
        raise ValueError("Prediction package does not match the selected document.")
    IntermediateAnalysis.model_validate(prediction_package["analysis"])
    write_predicted_package(OUTPUT_PATH, prediction_package)
else:
    print("JSON write skipped. Set WRITE_PREDICTIONS = True when ready.")

## Sanity checks and output preview

When writing is enabled, reopen the exact file G2 will read and validate its source, document ID, and segment coverage. Print only document ID, counts, checkpoint names, and destination; never print the segment text or prediction labels.

In [10]:
if WRITE_PREDICTIONS:
    if OUTPUT_PATH is None:
        raise RuntimeError("Configure PREBI_DATA_ROOT before checking the saved artifact.")
    saved_package = json.loads(OUTPUT_PATH.read_text(encoding="utf-8"))
    if saved_package["document_id"] != selected_document_id:
        raise ValueError("Saved prediction document ID does not match the selected document.")
    saved_analysis = IntermediateAnalysis.model_validate(saved_package["analysis"])
    if saved_analysis.source != "predicted":
        raise ValueError("Saved analysis must have source='predicted'.")
    make_generation_request(condition="G2", document=document, rubric=EXPERT_RUBRIC, analysis=saved_analysis)
    print(
        f"Ready for G2: document={selected_document_id}, segments={len(saved_analysis.segments)}, "
        f"checkpoints={list(checkpoints)}, file={OUTPUT_PATH}"
    )
else:
    print("No artifact to preview; inference and JSON writing are opt-in.")

Ready for G2: document=102, segments=47, checkpoints=['scope', 'direct-situationserfassung', 'direct-analyse', 'direct-konsequenzen'], file=/Users/quynguyen/Documents/CodingProjects/prebi_data/g2/predicted-analysis.json
